# Zadanie 2: optymalizacja z ograniczeniami

Termin realizacji: 31 marca 2025

Wybierz funkcję testową wykorzystaną w zadaniu 1.

Zadanie do oddania przez MS Teams. Do oddania: kod oraz krótkie sprawozdanie w PDF (można na przykład przy użyciu `quarto render notebook.ipynb --to pdf`).

## Na 3.0

Do realizacji:

1. Dodaj ograniczenie postaci $x_1^2 + x_2 + b = 0$ ze stałą $b$ dopasowaną w taki sposób, aby żadne minimum lokalne (przynajmniej w zakresie w którym losowany jest punkt początkowy) nie spełniają ograniczenia.
2. Zaimplementuj metodę funkcji kary do rozwiązania tego problemu.
3. Wylosuj 10 punktów z dziedziny przeszukiwania z tabelki. Dla każdego z nich przeprowadź 100 kroków optymalizacji metodą największego spadku ze stałym krokiem. Narysuj wykres zależności wartości funkcji optymalizowanej od kroku optymalizacji.
4. Przeprowadź procedurę dla kilku kroków. Spróbuj zilustrować brak zbieżności, szybką zbieżność i powolną zbieżność.

## Na 4.0

Do realizacji:

1. Punkty z zadania na 3.0.
2. Zamień metodę największego spadku na metodę gradientów sprzężonych.

## Na 5.0

Do realizacji:

1. Punkty z zadania na 4.0.
2. Wykonaj benchmarking metody z użyciem `BenchmarkTools.jl`. Zanotuj czasy działania wywołań optymalizacji oraz liczbę alokacji. Spróbuj zoptymalizować działanie funkcji korzystając wymienionych tu rad: [Julia performance tips](https://docs.julialang.org/en/v1/manual/performance-tips/). W sprawozdaniu napisz jakie zmiany wykonane i jak wpłynęły na czas działania programu.


In [4]:
using LinearAlgebra
using Plots
using Random

# 1. Funkcja z zad.1

In [5]:

function f(vec)
    x, y = vec[1], vec[2]
    return 2 * x^2 - 1.05 * x^4 + x^6 / 6 + x*y + y^2
end

function f_grad(vec)
    x, y = vec[1], vec[2]
    return [4 * x - 4.2 * x^3 + x^5 + y, x + 2 * y]
end

f_grad (generic function with 1 method)

In [6]:
function steepest_gradient_descent(cost, grad, x0, α, γ; max_iter=1000, tol=1e-8)
    θ = copy(x0)
    f_values = []
    storage = zeros(length(θ))
    for i in 1:max_iter
        value_start = cost(θ)
        storage = grad(θ)
        norm_storage = norm(storage)
        if norm_storage == 0
            break
        end
        θ_new = θ - storage .* (α / norm_storage)
        value_stop = cost(θ_new)
        push!(f_values, value_start)
        if abs(value_stop - value_start) < tol
            break
        else
            θ = θ_new
            α *= γ
        end
    end
    return θ, f_values
end

steepest_gradient_descent (generic function with 1 method)

In [7]:
function rand_uniform(a, b)
    return rand() * (b-a) + a
end

# lista z 10 wylosowanymi punktami
points = [[rand_uniform(-5, 5), rand_uniform(-5, 5)] for i in 1:10]
points

10-element Vector{Vector{Float64}}:
 [4.266944723499819, -3.698126376236602]
 [4.145291552097184, -2.612051048956323]
 [4.652663381318918, 0.07378703319873647]
 [3.6168846790450377, -3.381941199524227]
 [0.38072561758923573, 3.4380868061435823]
 [4.7804379055557025, 4.25587213931572]
 [3.3455331003456017, -3.211011576960753]
 [2.033495925470933, 0.07367091739320841]
 [4.35601143008534, -2.490471447976658]
 [2.3923920726674126, 1.0465483033724823]

In [ ]:
function g(vec)
    b = 4
    x, y = vec[1], vec[2]
    return x^2 + y + 4
end

function g_grad(vec)
    x, y = vec[1], vec[2]
    return [2 * x, 1]
end

function p(vec)
    return g(vec)^2
end

function p_grad(vec)
    return 2 * g(vec) * g_grad(vec)
end

function fp(vec, ρ)
    return f(vec) + ρ * p(vec)
end

function fp_grad(vec, ρ)
    return f_grad(vec) + ρ * p_grad(vec)
end

p_grad (generic function with 1 method)

In [ ]:
function penalty_method(fp_1, fp_grad_1, x0, k_max, ρ=1.0, γ=2.0, atol=1e-8)
    x = x0
    for k in 1 : k_max
        x = steepest_gradient_descent(fp_1, fp_grad_1, x, 0.1, 0.99)
        ρ *= γ
        if p(x) < atol
            return x
        end
    end
    return x
end

    

penalty_method (generic function with 4 methods)

In [23]:
penalty_method(fp, fp_grad, [0, 0], 16)

MethodError: MethodError: objects of type Float64 are not callable
The object of type `Float64` exists, but no method is defined for this combination of argument types when trying to treat it as a callable object.
Maybe you forgot to use an operator such as *, ^, %, / etc. ?